In [1]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn import preprocessing, metrics
import gc
import joblib
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR_PATH = '/kaggle/input/competitions/m5-forecasting-accuracy/'

def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics: 
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

def read_data():
    sell_prices_df = pd.read_csv(INPUT_DIR_PATH + 'sell_prices.csv')
    sell_prices_df = reduce_mem_usage(sell_prices_df)
    
    calendar_df = pd.read_csv(INPUT_DIR_PATH + 'calendar.csv')
    calendar_df = reduce_mem_usage(calendar_df)
    
    sales_train_validation_df = pd.read_csv(INPUT_DIR_PATH + 'sales_train_validation.csv')
    
    submission_df = pd.read_csv(INPUT_DIR_PATH + 'sample_submission.csv')
    return sell_prices_df, calendar_df, sales_train_validation_df, submission_df

def encode_categorical(df, cols):
    for col in cols:
        le = preprocessing.LabelEncoder()
        not_null = df[col][df[col].notnull()]
        df[col] = pd.Series(le.fit_transform(not_null), index=not_null.index)
    return df

def melt_and_merge(calendar, sell_prices, sales_train_validation, submission, nrows = 55000000, merge = False):
    sales_train_validation = pd.melt(sales_train_validation, id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], var_name = 'day', value_name = 'demand')
    sales_train_validation = reduce_mem_usage(sales_train_validation)
    sales_train_validation = sales_train_validation.iloc[-nrows:,:]

    test1_rows = [row for row in submission['id'] if 'validation' in row]
    test2_rows = [row for row in submission['id'] if 'evaluation' in row]
    test1 = submission[submission['id'].isin(test1_rows)]
    test2 = submission[submission['id'].isin(test2_rows)]

    test1.columns = ['id', 'd_1914', 'd_1915', 'd_1916', 'd_1917', 'd_1918', 'd_1919', 'd_1920', 'd_1921', 'd_1922', 'd_1923', 'd_1924', 'd_1925', 'd_1926', 'd_1927', 'd_1928', 'd_1929', 'd_1930', 'd_1931', 'd_1932', 'd_1933', 'd_1934', 'd_1935', 'd_1936', 'd_1937', 'd_1938', 'd_1939', 'd_1940', 'd_1941']
    test2.columns = ['id', 'd_1942', 'd_1943', 'd_1944', 'd_1945', 'd_1946', 'd_1947', 'd_1948', 'd_1949', 'd_1950', 'd_1951', 'd_1952', 'd_1953', 'd_1954', 'd_1955', 'd_1956', 'd_1957', 'd_1958', 'd_1959', 'd_1960', 'd_1961', 'd_1962', 'd_1963', 'd_1964', 'd_1965', 'd_1966', 'd_1967', 'd_1968', 'd_1969']

    product = sales_train_validation[['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']].drop_duplicates()

    test2['id'] = test2['id'].str.replace('_evaluation','_validation')
    test1 = test1.merge(product, how = 'left', on = 'id')
    test2 = test2.merge(product, how = 'left', on = 'id')
    test2['id'] = test2['id'].str.replace('_validation','_evaluation')

    test1 = pd.melt(test1, id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], var_name = 'day', value_name = 'demand')
    test2 = pd.melt(test2, id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], var_name = 'day', value_name = 'demand')

    sales_train_validation['part'] = 'train'
    test1['part'] = 'test1'
    test2['part'] = 'test2'

    data = pd.concat([sales_train_validation, test1, test2], axis = 0)
    del sales_train_validation, test1, test2

    data = data[data['part'] != 'test2']

    if merge:
        data = pd.merge(data, calendar, how = 'left', left_on = ['day'], right_on = ['d'])
        data.drop(['d'], inplace = True, axis = 1)
        data = data.merge(sell_prices, on = ['store_id', 'item_id', 'wm_yr_wk'], how = 'left')
    
    gc.collect()
    return data

def simple_fe(data):
    data['date'] = pd.to_datetime(data['date'])

    for val in [28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]:
        data[f"shift_t{val}"] = data.groupby(["id"])["demand"].transform(lambda x: x.shift(val))

    for val in [365, 366, 367]:
        data[f"shift_t{val}"] = data.groupby(["id"])["demand"].transform(lambda x: x.shift(val))

    for val in [7, 14, 28, 30, 60, 90, 180, 365]:
        data[f"rolling_std_t{val}"] = data.groupby(["id"])["demand"].transform(lambda x: x.shift(28).rolling(val).std())
        data[f"rolling_mean_t{val}"] = data.groupby(["id"])["demand"].transform(lambda x: x.shift(28).rolling(val).mean())
        data[f"rolling_min_t{val}"] = data.groupby(["id"])["demand"].transform(lambda x: x.shift(28).rolling(val).min())
        data[f"rolling_max_t{val}"] = data.groupby(["id"])["demand"].transform(lambda x: x.shift(28).rolling(val).max())
        data[f"rolling_range_t{val}"] = data[f"rolling_max_t{val}"] - data[f"rolling_min_t{val}"]

    data["rolling_skew_t30"] = data.groupby(["id"])["demand"].transform( lambda x: x.shift(28).rolling(30).skew())
    data["rolling_kurt_t30"] = data.groupby(["id"])["demand"].transform(lambda x: x.shift(28).rolling(30).kurt())

    data['lag_price_t1'] = data.groupby(['id'])['sell_price'].transform(lambda x: x.shift(1))
    data['price_change_t1'] = (data['lag_price_t1'] - data['sell_price']) / (data['lag_price_t1'])

    for val in [7, 28, 30]:
        data[f'lag_price_t{val}'] = data.groupby(['id'])['sell_price'].transform(lambda x: x.shift(val))

    data['rolling_price_max_t365'] = data.groupby(['id'])['sell_price'].transform(lambda x: x.shift(1).rolling(365).max())
    data['price_change_t365'] = (data['rolling_price_max_t365'] - data['sell_price']) / (data['rolling_price_max_t365'])

    for val in [7, 14, 30, 60]:
        data[f"rolling_price_std_t{val}"] = data.groupby(['id'])['sell_price'].transform(lambda x: x.rolling(val).std())
        data[f"rolling_price_mean_t{val}"] = data.groupby(['id'])['sell_price'].transform(lambda x: x.rolling(val).mean())

    data.drop(['rolling_price_max_t365', 'lag_price_t1'], inplace = True, axis = 1)

    # Removed "week" from this list
    attrs = ["year", "quarter", "month", "day", "dayofweek", "is_year_end", "is_year_start", "is_quarter_end", "is_quarter_start", "is_month_end","is_month_start"]
    
    for attr in attrs:
        dtype = np.int16 if attr == "year" else np.int8
        data[attr] = getattr(data['date'].dt, attr).astype(dtype)
        
    # Add the week column using the updated Pandas 2.0+ syntax
    data["week"] = data['date'].dt.isocalendar().week.astype(np.int8)
    
    data["is_weekend"] = data["dayofweek"].isin([5, 6]).astype(np.int8)

    return data

def run_lgb(data):
    x_train = data[data['date'] <= '2016-03-27'].copy()
    y_train = x_train['demand']
    x_val = data[(data['date'] > '2016-03-27') & (data['date'] <= '2016-04-24')].copy()
    y_val = x_val['demand']
    test = data[(data['date'] > '2016-04-24')].copy()

    del data
    gc.collect()

    print("--- Training initial model for feature importance ---")
    cols_to_drop = ['id', 'demand', 'date', 'part']
    initial_features = [col for col in x_train.columns if col not in cols_to_drop]
    numeric_features = x_train[initial_features].select_dtypes(include=np.number).columns.tolist()
    initial_features = numeric_features

    # Explicitly define categorical features for LightGBM
    cat_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
    active_cats = [col for col in cat_cols if col in initial_features]

    train_set_initial = lgb.Dataset(x_train[initial_features], y_train, categorical_feature=active_cats)
    val_set_initial = lgb.Dataset(x_val[initial_features], y_val, categorical_feature=active_cats)

    params_initial = {
        'metric': 'rmse',
        'objective': 'poisson',
        'n_jobs': -1,
        'seed': 20,
        'learning_rate': 0.05,
        'num_leaves': 128,
        'min_data_in_leaf': 100,
        'lambda_l1': 0.1,
        'lambda_l2': 0.1,
        'bagging_fraction': 0.7,
        'bagging_freq': 1,
        'feature_fraction': 0.7,
        'verbose': -1
    }

    model_initial = lgb.train(
        params_initial,
        train_set_initial,
        num_boost_round = 1000,
        valid_sets = [val_set_initial],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=0)
        ]
    )
    feature_importances = pd.DataFrame({'feature': initial_features, 'importance': model_initial.feature_importance(importance_type='gain')})
    feature_importances = feature_importances.sort_values(by='importance', ascending=False).reset_index(drop=True)

    print("\n--- Training final model with selected features ---")
    selected_features = feature_importances[feature_importances['importance'] > 0]['feature'].tolist()
    selected_features = [f for f in selected_features if f in x_train.columns and np.issubdtype(x_train[f].dtype, np.number)]
    
    # Update active categorical features based on selection
    active_cats_final = [col for col in cat_cols if col in selected_features]

    train_set_final = lgb.Dataset(x_train[selected_features], y_train, categorical_feature=active_cats_final)
    val_set_final = lgb.Dataset(x_val[selected_features], y_val, categorical_feature=active_cats_final)

    params_final = {
        'metric': 'rmse',
        'objective': 'poisson',
        'n_jobs': -1,
        'seed': 20,
        'learning_rate': 0.05,
        'num_leaves': 64,
        'min_data_in_leaf': 20,
        'lambda_l1': 0.1,
        'lambda_l2': 0.1,
        'bagging_fraction': 0.7,
        'bagging_freq': 1,
        'feature_fraction': 0.7,
    }

    model_final = lgb.train(
        params_final,
        train_set_final,
        num_boost_round = 2000,
        valid_sets = [train_set_final, val_set_final],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=100)
        ]
    )
    joblib.dump(model_final, 'lgbm_final_model.sav')

    # ---------------------------------------------------------
    # CALCULATE ITEM-LEVEL RMSE & EXPORT FOR THE RESEARCH PAPER
    # ---------------------------------------------------------
    print("\n--- Calculating Item-Level Metrics ---")
    val_pred = model_final.predict(x_val[selected_features], num_iteration=model_final.best_iteration)
    
    x_val_eval = x_val.copy()
    x_val_eval['predicted_demand'] = val_pred
    x_val_eval['actual_demand'] = y_val
    
    # (Predicted - Actual)^2
    x_val_eval['squared_error'] = (x_val_eval['predicted_demand'] - x_val_eval['actual_demand']) ** 2
    
    # Group by 'id', calculate Mean Squared Error, then take Square Root
    item_rmse_df = x_val_eval.groupby('id')['squared_error'].mean().apply(np.sqrt).reset_index()
    item_rmse_df.rename(columns={'squared_error': 'lgbm_rmse'}, inplace=True)
    
    # Save to CSV to compare against Croston's later
    item_rmse_df.to_csv('lgbm_item_level_rmse.csv', index=False)
    
    print(f"Successfully saved item-level RMSE for {len(item_rmse_df)} products to 'lgbm_item_level_rmse.csv'")
    
    # Global metrics printout
    global_rmse = np.sqrt(metrics.mean_squared_error(val_pred, y_val))
    r2 = metrics.r2_score(val_pred, y_val)
    mae = metrics.mean_absolute_error(val_pred, y_val)
    print(f'Our final global val rmse score is {global_rmse}')
    print(f'Our final global val mae score is {mae}')
    print(f'Our final global val r2 score is {r2}')
    # ---------------------------------------------------------

    y_pred = model_final.predict(test[selected_features], num_iteration=model_final.best_iteration)
    test['demand'] = y_pred
    return test

def predict(test, submission):
    predictions = test[['id', 'date', 'demand']]
    predictions = pd.pivot(predictions, index = 'id', columns = 'date', values = 'demand').reset_index()
    predictions.columns = ['id'] + ['F' + str(i + 1) for i in range(28)]

    evaluation_rows = [row for row in submission['id'] if 'evaluation' in row]
    evaluation = submission[submission['id'].isin(evaluation_rows)]

    validation = submission[['id']].merge(predictions, on = 'id')
    final = pd.concat([validation, evaluation])
    final.to_csv('submission.csv', index = False)

def transform_train_and_eval(data):
    data = simple_fe(data)
    print(f"Data shape after feature engineering: {data.shape}")
    data = reduce_mem_usage(data)
    test = run_lgb(data)
    predict(test, submission_df)

# --- EXECUTION ---
sell_prices_df, calendar_df, sales_train_validation_df, submission_df = read_data() 

calendar_df = encode_categorical(calendar_df, ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]).pipe(reduce_mem_usage)
sales_train_validation_df = encode_categorical(sales_train_validation_df, ["item_id", "dept_id", "cat_id", "store_id", "state_id"]).pipe(reduce_mem_usage)
sell_prices_df = encode_categorical(sell_prices_df, ["item_id", "store_id"]).pipe(reduce_mem_usage)

# WARNING: 15,000,000 rows with 365-day rolling features requires massive RAM. 
# It is highly recommended to start with nrows = 500000 or nrows = 1000000 to test locally first!
nrows = 15000000 
data = melt_and_merge(calendar_df, sell_prices_df, sales_train_validation_df, submission_df, nrows = nrows, merge = True)

transform_train_and_eval(data)

Mem. usage decreased to 130.48 Mb (37.5% reduction)
Mem. usage decreased to  0.12 Mb (41.9% reduction)
Mem. usage decreased to  0.08 Mb (36.9% reduction)
Mem. usage decreased to 94.01 Mb (78.9% reduction)
Mem. usage decreased to 45.67 Mb (65.0% reduction)
Mem. usage decreased to 1335.01 Mb (0.0% reduction)
Data shape after feature engineering: (15853720, 106)
Mem. usage decreased to 3447.20 Mb (64.2% reduction)
--- Training initial model for feature importance ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[391]	valid_0's rmse: 2.13107

--- Training final model with selected features ---
Training until validation scores don't improve for 200 rounds
[100]	training's rmse: 2.28234	valid_1's rmse: 2.19193
[200]	training's rmse: 2.21964	valid_1's rmse: 2.15845
[300]	training's rmse: 2.18067	valid_1's rmse: 2.14788
[400]	training's rmse: 2.15433	valid_1's rmse: 2.14078
[500]	training's rmse: 2.12767	valid_1's rmse: 2.13425
[600]	training